In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import plotly.express as px


In [2]:
data = pd.read_excel('bda_dataset4_3.xlsx')

print(data.head())
data.info()


   anime_id                                               name  \
0     32281                                     Kimi no Na wa.   
1      5114                   Fullmetal Alchemist: Brotherhood   
2     28977                                          GintamaВ°   
3     32935  Haikyuu!!: Karasuno Koukou VS Shiratorizawa Ga...   
4     11061                             Hunter x Hunter (2011)   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3             Comedy, Drama, School, Shounen, Sports     TV       10    9.15   
4            Action, Adventure, Shounen, Super Power     TV      148    9.13   

   members  
0   200630  
1   793665  
2   114262  
3    93351  
4   425855  
<class 'pandas.core.frame.DataFrame'>
RangeI

In [ ]:
data['genre'] = data['genre'].astype(str).str.lower().str.replace(';', ',').str.replace(' ', '')
genre_counts = Counter()
data['genre'].str.split(',').apply(genre_counts.update)
top_genres = dict(genre_counts.most_common(10))
fig = px.bar(
    x=list(top_genres.values()),
    y=list(top_genres.keys()),
    orientation='h',
    labels={'x': 'Кількість аніме', 'y': 'Жанр'},
    title='Кількість аніме у 10 найпопулярніших жанрах'
)
fig.show()


In [4]:
# TF-IDF векторизаці
tfidf = TfidfVectorizer(tokenizer=lambda x: x.split(','), stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['genre'])

# Косинус
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)


C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\feature_extraction\text.py:517: UserWarning:

The parameter 'token_pattern' will not be used since 'tokenizer' is not None'



In [ ]:
def recommend_anime(title, cosine_sim=cosine_sim):
    if title not in data['name'].values:
        return f"Аніме '{title}' не знайдено ."
    
    idx = data[data['name'] == title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]  
    anime_indices = [i[0] for i in sim_scores]
    return data[['name', 'genre']].iloc[anime_indices]


In [8]:
anime_title = "Naruto"  
recommended = recommend_anime(anime_title)
print(f"10 найбільш подібних до '{anime_title}':")
print(recommended)


10 найбільш подібних до 'Naruto':
                                                   name  \
601                                  Naruto: Shippuuden   
818                                              Naruto   
1076  Boruto: Naruto the Movie - Naruto ga Hokage ni...   
1311                                        Naruto x UT   
1436        Naruto: Shippuuden Movie 4 - The Lost Tower   
1534  Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...   
2401               Naruto Shippuuden: Sunny Side Battle   
2921  Naruto Soyokazeden Movie: Naruto to Mashin to ...   
7396                            Kyutai Panic Adventure!   
764          Naruto: Shippuuden Movie 6 - Road to Ninja   

                                                genre  
601      action,comedy,martialarts,shounen,superpower  
818      action,comedy,martialarts,shounen,superpower  
1076     action,comedy,martialarts,shounen,superpower  
1311     action,comedy,martialarts,shounen,superpower  
1436     action,comedy,martialarts,s